# RISKY: 30-day hospital readmission, complete data science pipeline
**Author:** Karina Roade · QH0636 Data Science (resit)

**Research question.** How well can supervised machine learning models identify diabetic hospital encounters that are followed by readmission within 30 days, and what trade-offs between detection and false alarms do they involve?

**Objectives.**
1. Clean and prepare a messy healthcare dataset without leaking test information.
2. Explore patterns linked to early readmission with appropriate visualisations.
3. Train and justify three classification algorithms.
4. Evaluate and critically compare them with metrics suited to an imbalanced target.

**Dataset.** Diabetes 130-US Hospitals (1999–2008), UCI Machine Learning Repository: 101,766 encounters, 50 columns.

**Response variable.** The dataset has **one** response variable, `readmitted`, with **three** classes: `<30` (readmitted within 30 days), `>30` (readmitted after 30 days) and `NO`. This project models a **binary** target derived from it: `early_readmission = 1` if `readmitted == "<30"`, otherwise `0`.

**How to run.** Open this notebook from the project folder (the folder containing `data/` and `outputs/`) and choose *Run All*. It needs the packages in `requirements.txt`. The standalone scripts (`train_models.py` and the others, see `README.md`) produce the saved files in `outputs/`; this notebook repeats the same steps with every result shown inline, and checks that its numbers match the saved outputs.

## 1. Setup

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder

ROOT = Path.cwd()                       # the notebook is run from the project folder
DATA = ROOT / 'data' / 'diabetic_data.csv'
OUT = ROOT / 'outputs'
assert DATA.exists(), f'Run this notebook from the project folder; {DATA} not found'
pd.set_option('display.max_columns', 60)
print('Project folder:', ROOT)

## 2. Load the original data
`?` marks a missing value in this file. `keep_default_na=False` stops pandas from turning the text `None` into a missing value: in `max_glu_serum` and `A1Cresult`, `None` means *the test was not performed*, which is information, not missingness.

In [ ]:
raw = pd.read_csv(DATA, na_values=['?'], keep_default_na=False, low_memory=False)
print('Shape:', raw.shape)
numeric_cols = raw.select_dtypes(include='number').columns.tolist()
categorical_cols = [c for c in raw.columns if c not in numeric_cols]
print(f'{len(numeric_cols)} numerical columns, {len(categorical_cols)} categorical columns')
raw.head()

## 3. Response variable: one column, three classes, mapped to a binary target

In [ ]:
print(raw['readmitted'].value_counts().reindex(['<30', '>30', 'NO']).to_string())
y_all = (raw['readmitted'] == '<30').astype(int)
print(f'\nBinary target early_readmission: {y_all.sum():,} positives out of {len(y_all):,} '
      f'({y_all.mean():.2%}).')
print(f'Predicting "no early readmission" for everyone would be {1 - y_all.mean():.2%} accurate '
      'while detecting no positive case, so accuracy alone is misleading.')

## 4. Data quality and cleaning
The cleaning steps live in `make_clean_dataset.py` so that the notebook, the scripts and the saved cleaned file all use exactly the same rules. Missing values are **not** imputed here: imputers are fitted inside the model pipelines on training data only, to avoid leaking test information.

In [ ]:
from make_clean_dataset import clean

missing = raw.isna().sum()
missing = (pd.DataFrame({'missing_count': missing, 'missing_%': (missing / len(raw) * 100).round(2)})
           .query('missing_count > 0').sort_values('missing_count', ascending=False))
print('Missing values after mapping "?" (before cleaning):')
display(missing)

print('Lab tests: "None" = not performed (kept as a category)')
for col in ['max_glu_serum', 'A1Cresult']:
    print(col, raw[col].value_counts().to_dict())
print('gender:', raw['gender'].value_counts().to_dict())

df, cleaning_log = clean(raw)
display(cleaning_log)
df.to_csv(ROOT / 'data' / 'diabetic_data_cleaned.csv', index=False)
print('Cleaned dataset saved: data/diabetic_data_cleaned.csv', df.shape)

## 5. Exploratory data analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
counts = raw['readmitted'].value_counts().reindex(['<30', '>30', 'NO'])
axes[0].bar(counts.index, counts.values)
axes[0].set_title('Original readmission classes'); axes[0].set_ylabel('Hospital encounters')
top = missing.head(8).iloc[::-1]
axes[1].barh(top.index, top['missing_%'])
axes[1].set_title('Missing values after mapping "?" (%)'); axes[1].set_xlabel('% missing')
plt.tight_layout(); plt.show()

**Reading.** Only about one encounter in nine is an early readmission (class imbalance). Weight is almost entirely missing and is dropped; medical specialty and payer code are about half and two-fifths missing and are imputed inside the pipelines.

In [ ]:
def rate_table(series):
    t = df.groupby(series, observed=True)['early_readmission'].agg(encounters='size', rate='mean')
    t['rate_%'] = (t.pop('rate') * 100).round(2)
    return t

prior = rate_table(pd.cut(df['number_inpatient'], [-1, 0, 1, 2, 100], labels=['0', '1', '2', '3+']))
age = rate_table(df['age'])
stay = rate_table(df['time_in_hospital'])

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
axes[0].bar(prior.index.astype(str), prior['rate_%']); axes[0].set_title('By prior inpatient visits')
axes[1].bar(age.index.astype(str), age['rate_%']); axes[1].set_title('By age band'); axes[1].tick_params(axis='x', rotation=45)
axes[2].plot(stay.index, stay['rate_%'], marker='o'); axes[2].set_title('By length of stay (days)')
for ax in axes: ax.set_ylabel('Early readmission (%)')
plt.tight_layout(); plt.show()
display(prior.T); display(age.T); display(stay.T)

**Reading.** The rate rises from about 8% with no prior inpatient visit to about 26% with three or more, the strongest pattern in the data. Age bands differ only modestly (the youngest bands are small), and longer stays are associated with higher rates up to about 10 days. These are associations, not causal effects: illness severity is not fully measured and can explain both.

## 6. Modelling set-up
* **Features:** all columns except the target, the original `readmitted`, the identifiers `encounter_id` and `patient_nbr`, and `weight` (already dropped).
* **Patient-grouped split:** 71,518 patients have 101,766 encounters, so encounters of one patient must not appear in both training and test sets. `GroupShuffleSplit` keeps 20% of **patients** for testing.
* **Pipelines:** imputation, scaling and encoding are fitted on training data only. Numerical: median imputation (+ scaling for the linear-preprocessing models). Categorical: most-frequent imputation, then one-hot encoding (rare categories under 1,000 grouped) or ordinal encoding for gradient boosting.
* **Algorithms:**
  * *Logistic Regression*: interpretable linear baseline, balanced class weights.
  * *Random Forest*: averages many decision trees to capture non-linear effects, balanced class weights.
  * *HistGradientBoostingClassifier*: scikit-learn's histogram-based gradient boosting (the same family as XGBoost/LightGBM); builds trees sequentially to model feature interactions. No class weighting.

In [ ]:
y = df['early_readmission']
X = df.drop(columns=['early_readmission', 'readmitted', 'encounter_id', 'patient_nbr'])
cat = X.select_dtypes(include=['object', 'string']).columns.tolist()
num = [c for c in X.columns if c not in cat]

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=.20, random_state=42).split(X, y, groups=df['patient_nbr']))
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y.iloc[tr], y.iloc[te]
assert set(df['patient_nbr'].iloc[tr]).isdisjoint(df['patient_nbr'].iloc[te]), 'patient overlap!'
print(f'{len(num)} numerical and {len(cat)} categorical features')
print(f'Train: {len(tr):,} encounters (prevalence {ytr.mean():.4f}) | Test: {len(te):,} encounters (prevalence {yte.mean():.4f})')
print('No patient appears in both sets.')

linear_prep = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('onehot', OneHotEncoder(handle_unknown='ignore', min_frequency=1000))]), cat)])
tree_prep = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                      ('ordinal', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat)])

def make_models():
    return {
        'Logistic Regression': Pipeline([('prep', linear_prep), ('model', LogisticRegression(max_iter=1500, class_weight='balanced', random_state=42))]),
        'Random Forest': Pipeline([('prep', linear_prep), ('model', RandomForestClassifier(n_estimators=35, max_depth=12, min_samples_leaf=8, class_weight='balanced_subsample', random_state=42, n_jobs=2))]),
        'Gradient Boosting': Pipeline([('prep', tree_prep), ('model', HistGradientBoostingClassifier(max_iter=65, max_leaf_nodes=12, l2_regularization=1.0, random_state=42))]),
    }

## 7. Training and evaluation on held-out patients (threshold 0.50)

In [ ]:
results, probs = [], {}
for name, pipe in make_models().items():
    pipe.fit(Xtr, ytr)
    p = pipe.predict_proba(Xte)[:, 1]; pred = (p >= 0.5).astype(int)
    probs[name] = p
    results.append({'model': name, 'accuracy': accuracy_score(yte, pred), 'balanced_accuracy': balanced_accuracy_score(yte, pred),
                    'precision': precision_score(yte, pred, zero_division=0), 'recall': recall_score(yte, pred, zero_division=0),
                    'f1': f1_score(yte, pred, zero_division=0), 'roc_auc': roc_auc_score(yte, p),
                    'pr_auc_average_precision': average_precision_score(yte, p), 'brier_score': brier_score_loss(yte, p)})
results = pd.DataFrame(results)
display(results.round(3))

# Reproducibility check against the files written by train_models.py
saved = pd.read_csv(OUT / 'model_comparison.csv')
saved_preds = pd.read_csv(OUT / 'test_predictions.csv')
same_metrics = np.allclose(results.drop(columns='model').to_numpy(), saved.drop(columns='model').to_numpy(), atol=1e-9)
same_preds = all(np.allclose(probs[m], saved_preds[m], atol=1e-9) for m in probs)
print('Matches outputs/model_comparison.csv:', same_metrics, '| matches outputs/test_predictions.csv:', same_preds)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
for ax, (name, p) in zip(axes, probs.items()):
    ConfusionMatrixDisplay(confusion_matrix(yte, (p >= 0.5).astype(int)),
                           display_labels=['No early', '<30 days']).plot(ax=ax, values_format='d', cmap='Blues', colorbar=False)
    ax.set_title(name)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for name, p in probs.items():
    RocCurveDisplay.from_predictions(yte, p, name=name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(yte, p, name=name, ax=axes[1])
axes[0].set_title('ROC, held-out patients'); axes[1].set_title('Precision-recall, held-out patients')
plt.tight_layout(); plt.show()

**Reading.** Gradient Boosting ranks encounters best (highest ROC-AUC and average precision) but, at the 0.50 threshold, detects only 9 of 2,151 early readmissions; its high accuracy mostly reflects the majority class. Logistic Regression and Random Forest detect about half of the readmissions but with low precision (about one flagged encounter in six is a true readmission). No model gives a clinically acceptable balance on its own.

## 8. Threshold sensitivity (descriptive only, no threshold is selected on the test set)

In [ ]:
rows = []
for name, p in probs.items():
    for t in [0.10, 0.20, 0.30, 0.40, 0.50]:
        z = (p >= t).astype(int); tn, fp, fn, tp = confusion_matrix(yte, z, labels=[0, 1]).ravel()
        rows.append({'model': name, 'threshold': t, 'precision': precision_score(yte, z, zero_division=0),
                     'recall': recall_score(yte, z, zero_division=0), 'f1': f1_score(yte, z, zero_division=0),
                     'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn})
thresholds = pd.DataFrame(rows)
display(thresholds.round(3))
print('Matches outputs/threshold_sensitivity.csv:',
      np.allclose(thresholds.drop(columns='model').to_numpy(float),
                  pd.read_csv(OUT / 'threshold_sensitivity.csv').drop(columns='model').to_numpy(float)))

## 9. Calibration check (probabilities are **not** recalibrated)

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5))
for name, p in probs.items():
    obs, est = calibration_curve(yte, p, n_bins=10, strategy='quantile')
    ax.plot(est, obs, marker='o', label=name)
ax.plot([0, 1], [0, 1], '--', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed readmission frequency')
ax.set_title('Reliability diagram (10 quantile bins)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()
prev = yte.mean()
cal = results[['model', 'brier_score']].assign(mean_predicted=[probs[m].mean() for m in results['model']], observed_rate=prev)
display(cal.round(3))
print(f'Reference Brier score for always predicting the prevalence ({prev:.3f}): {prev * (1 - prev):.3f}')

**Reading.** Gradient Boosting's probabilities follow observed rates closely. Logistic Regression and Random Forest strongly overestimate risk (mean prediction about 0.46 against an observed 0.107), as expected after balanced class weighting, so their probabilities should not be read as risks without recalibration.

## 10. Robustness: patient-grouped 3-fold cross-validation on the training set only (takes a few minutes)

In [ ]:
Xtrain, ytrain = Xtr.reset_index(drop=True), ytr.reset_index(drop=True)
gtrain = df['patient_nbr'].iloc[tr].reset_index(drop=True)
cv_rows = []
for fold, (it, iv) in enumerate(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42).split(Xtrain, ytrain, groups=gtrain), 1):
    for name, model in make_models().items():
        model.fit(Xtrain.iloc[it], ytrain.iloc[it])
        cv_rows.append({'fold': fold, 'model': name, 'roc_auc': roc_auc_score(ytrain.iloc[iv], model.predict_proba(Xtrain.iloc[iv])[:, 1])})
cv = pd.DataFrame(cv_rows).groupby('model')['roc_auc'].agg(['mean', 'std']).round(3)
display(cv)

**Reading.** Cross-validated ROC-AUC values are within about 0.005 of the test-set values, with low variation between folds, so the ranking of the models is stable rather than an artefact of one split.

## 11. Summary against the objectives
1. **Cleaning:** placeholders, an uninformative column, invalid gender codes and repeat patients were handled; imputation is fitted on training data only; the cleaned file is saved in `data/`.
2. **EDA:** prior inpatient use shows the strongest association with early readmission; age and length of stay show weaker patterns; all are associations only.
3. **Modelling:** three justified algorithms were trained in leakage-safe pipelines.
4. **Evaluation:** the models discriminate modestly (ROC-AUC 0.64–0.67). Their usefulness depends on the threshold and the relative cost of missed readmissions versus false alarms, which is a clinical decision. Results are reproducible and match the saved outputs.